# PS5 world view (SLAM) on Colab
Puts the worker's head (as a camera), gaze, body axis and both hands in one room-fixed 3D frame,
using the clip and its **existing PS4 results** (PS4 is not run again).

**Before you start**
1. Runtime → Change runtime type → **A100** (or G4). Use a fresh runtime, not the PS4 one.
2. In Google Drive (`MyDrive/mytronD/` by default):
   - `ps5_slam.zip` (the code),
   - the clip videos (any `Copy of ...` prefix is fine),
   - `ps4v4_results/<clip>/` for each clip, with `hands.jsonl`, `camera.json`, `head_rot.pkl`.
3. Run cells 1 → 4. Set the clip name(s) in cell 3.

Results: `MyDrive/mytronD/ps5_world/<clip>/` → `world_render.mp4`, `scene.rrd`, `world_hands.jsonl`,
`trajectory.csv`, `world_stats.json`.

*Data rule: outputs contain footage. Keep them private; delete all copies after the event.*

In [ ]:
# CELL 1: install ViPE (compiles its CUDA kernels, ~10-20 min) + dependencies
import torch, os
assert torch.cuda.is_available(), "switch the runtime to a GPU"
cc = torch.cuda.get_device_capability()
os.environ["TORCH_CUDA_ARCH_LIST"] = f"{cc[0]}.{cc[1]}"          # compile only for this GPU
print(torch.cuda.get_device_name(0), "| arch", os.environ["TORCH_CUDA_ARCH_LIST"], "| torch", torch.__version__)
!pip -q install ninja setuptools wheel
!pip -q install --no-build-isolation nvidia-vipe
!pip -q install rerun-sdk pandas pyyaml scipy opencv-python git+https://github.com/cvg/GeoCalib
try:
    import vipe; print("ViPE installed")
except Exception as e:
    print("ViPE NOT installed:", e, "-> cell 3 will use BACKEND='rotation' (head position fixed)")
import rerun as rr; print("rerun-sdk", rr.__version__, "(open scene.rrd with this same version)")

In [ ]:
# CELL 2: mount Drive, unpack the code, self-test (~1 min)
from google.colab import drive
drive.mount("/content/drive")
import zipfile, shutil, os, glob
DRIVE_DIR = "/content/drive/MyDrive/mytronD"

import re
NEED = (1, 3)                                     # code version with the partial-clip fix

def slam_version(z):
    # (major, minor) of the SLAM code in a zip, or None if the zip has no SLAM code
    try:
        zf = zipfile.ZipFile(z)
        names = zf.namelist()
        if not any(n.endswith("run_slam.py") for n in names):
            return None
        init = [n for n in names if n.endswith("ps5world/__init__.py")]
        m = re.search(r'VERSION = "(\d+)\.(\d+)"', zf.read(init[0]).decode()) if init else None
        return (int(m.group(1)), int(m.group(2))) if m else (1, 0)
    except zipfile.BadZipFile:
        return None

# pick the zip with the NEWEST CODE VERSION (Drive upload dates are not reliable; "ps5_slam (1).zip" etc. are fine)
zips = glob.glob(f"{DRIVE_DIR}/*.zip")
vers = {z: slam_version(z) for z in zips}
for z, v in sorted(vers.items(), key=lambda kv: kv[1] or (0, 0), reverse=True):
    print(f"  {os.path.basename(z):40s} {'SLAM code v%d.%d' % v if v else 'no SLAM code'}")
good = sorted([z for z, v in vers.items() if v], key=lambda z: (vers[z], os.path.getmtime(z)), reverse=True)
assert good, "no zip with run_slam.py in " + DRIVE_DIR + ": upload ps5_slam.zip from D:/personal/mytron/latest/"
ZIP = good[0]
assert vers[ZIP] >= NEED, (f"newest SLAM code in Drive is v{vers[ZIP][0]}.{vers[ZIP][1]} (old, crashes on "
    f"singular matrix). Upload the new ps5_slam.zip from D:/personal/mytron/latest/ to {DRIVE_DIR}, then rerun this cell.")
shutil.rmtree("/content/_src", ignore_errors=True); shutil.rmtree("/content/code", ignore_errors=True)
zipfile.ZipFile(ZIP).extractall("/content/_src")
root = os.path.dirname(sorted(glob.glob("/content/_src/**/run_slam.py", recursive=True), key=len)[0])
shutil.move(root, "/content/code")
print("code from", ZIP, "| version", vers[ZIP])
import sys; sys.path.insert(0, "/content/code")
for _m in [m for m in list(sys.modules) if m.startswith("ps5world")]: del sys.modules[_m]
!cd /content/code && python tests/test_world_synthetic.py 2>&1 | tail -1

In [ ]:
# CELL 3: settings + run
import os, glob, shutil, subprocess, sys
CLIPS_DIR   = f"{DRIVE_DIR}"                       # folder with the clip videos
PS4_DIR     = f"{DRIVE_DIR}/ps4v4_results"         # one PS4 result folder per clip
OUT_DIR     = f"{DRIVE_DIR}/ps5_world"             # results are copied here
CLIPS       = ["mytronlabs-000003-008"]            # clip name(s) without "Copy of" and .mp4
MAX_SECONDS = 60                                   # first try 60 s (~5-10 min); None = whole clip
SLAM_FPS    = 15
BACKEND     = "auto"                               # auto = ViPE if installed, else rotation-only

# which clips are ready (PS4 files present)
!cd /content/code && python run_slam.py --clips-dir "{CLIPS_DIR}" --ps4-dir "{PS4_DIR}" --list

# copy the chosen videos to local disk (much faster to read than Drive)
LOCAL_CLIPS = "/content/clips"; os.makedirs(LOCAL_CLIPS, exist_ok=True)
for name in CLIPS:
    src = [p for p in glob.glob(f"{CLIPS_DIR}/*.mp4") if os.path.basename(p).replace("Copy of ", "").strip() == f"{name}.mp4"]
    assert src, f"no video for {name} in {CLIPS_DIR}"
    dst = f"{LOCAL_CLIPS}/{name}.mp4"
    if not os.path.exists(dst):
        shutil.copy(src[0], dst)

LOCAL_OUT = "/content/ps5_world"
cmd = [sys.executable, "run_slam.py", "--clips-dir", LOCAL_CLIPS, "--ps4-dir", PS4_DIR, "--out-dir", LOCAL_OUT,
       "--clip", *CLIPS, "--backend", BACKEND, "--slam-fps", str(SLAM_FPS)]
if MAX_SECONDS:
    cmd += ["--max-seconds", str(MAX_SECONDS)]
print(" ".join(cmd))
# stream the log here (subprocess.run would hide it in Colab) and keep a copy
LOG = f"{LOCAL_OUT}/run_slam.log"; os.makedirs(LOCAL_OUT, exist_ok=True)
with subprocess.Popen(cmd, cwd="/content/code", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                      bufsize=1) as p, open(LOG, "w") as lf:
    for line in p.stdout:
        print(line, end=""); lf.write(line)
print("exit code:", p.returncode)

KEEP = ("world_render.mp4", "scene.rrd", "world_hands.jsonl", "world_hands_mesh.npz", "trajectory.csv",
        "world_stats.json", "run_slam.log")
for name in CLIPS:
    os.makedirs(f"{OUT_DIR}/{name}", exist_ok=True)
    for f in KEEP:
        if os.path.exists(f"{LOCAL_OUT}/{name}/{f}"):
            shutil.copy(f"{LOCAL_OUT}/{name}/{f}", f"{OUT_DIR}/{name}/{f}")
    shutil.copy(LOG, f"{OUT_DIR}/{name}/run_slam.log")
    print("saved:", f"{OUT_DIR}/{name}", os.listdir(f"{OUT_DIR}/{name}"))

In [ ]:
# CELL 4: check the numbers, watch the render
import json
from IPython.display import Video, display
for name in CLIPS:
    p = f"{LOCAL_OUT}/{name}/world_stats.json"
    if not os.path.exists(p):
        print(name, ": no result (see the log of cell 3)"); continue
    st = json.load(open(p))
    print(name, json.dumps({k: st.get(k) for k in ("backend", "up_method", "hand_scale_factor", "head_path_m",
          "rotation_1s_vs_ps4_median_deg", "left_hand_to_head_median_m", "right_hand_to_head_median_m",
          "seconds")}, indent=1))
    display(Video(f"{LOCAL_OUT}/{name}/world_render.mp4", embed=True, width=1000))
# look for: backend = vipe; rotation_1s_vs_ps4_median_deg <= ~2; hand-to-head 0.3-0.8 m

### Notes
- **Whole clip:** set `MAX_SECONDS = None` and run cell 3 again. A finished clip is skipped; delete its
  folder in `/content/ps5_world/` (or add `"--force"` to `cmd`) to redo it.
- **ViPE did not install:** set `BACKEND = "rotation"`: head turns, gaze, body and hands are right, head position is fixed.
- **3D scene:** download `scene.rrd`, then on your computer `pip install rerun-sdk==<version from cell 1>` and `rerun scene.rrd`.